# VLS Benchmark — Data & Experiment Visualization

This notebook covers:
1. **Dataset overview** — class balance, split sizes, protein coverage
2. **Ligand chemistry** — molecular weight, SMILES length, fingerprint density
3. **Model performance** — ROC-AUC / PR-AUC across train / val / test splits
4. **Generalization analysis** — train→test gap, per-metric comparison
5. **PDBbind comparison** — our results vs literature baselines

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
FIGSIZE = (10, 5)
COLORS = {"random_forest": "#4C72B0", "gradient_boosting": "#DD8452", "svm": "#55A868"}
SPLIT_COLORS = {"train": "#4C72B0", "val": "#DD8452", "test": "#55A868"}
SPLIT_MARKERS = {"1d": "o", "2d": "s"}
SPLIT_HATCHES = {"1d": "", "2d": "//"}

# ── paths ──────────────────────────────────────────────────────────────
ROOT = Path("../").resolve()
REGISTRY      = ROOT / "training_data_full/registry.csv"
MODELS_DIR    = ROOT / "benchmarks/02_training/trained_models"
MODELS_DIR_2D = ROOT / "benchmarks/02_training/trained_models_2d"
REPORT_CSV    = ROOT / "benchmarks/03_analysis/report.csv"

print("ROOT:", ROOT)
print("Registry exists:     ", REGISTRY.exists())
print("Models dir (1D):     ", MODELS_DIR.exists())
print("Models dir (2D):     ", MODELS_DIR_2D.exists())


## 1  Dataset Overview

In [ ]:
print("Loading registry...")
reg = pd.read_csv(REGISTRY)
print(f"Total rows: {len(reg):,}")
reg.head(2)

In [ ]:
# ── split × threshold breakdown ────────────────────────────────────────
summary = (
    reg.groupby(["similarity_threshold", "split"])
    .agg(
        n_total=("smiles", "count"),
        n_active=("is_active", "sum"),
        n_proteins=("uniprot_id", "nunique"),
        n_unique_smiles=("smiles", "nunique"),
    )
    .reset_index()
)
summary["pct_active"] = (summary["n_active"] / summary["n_total"] * 100).round(2)
summary

In [ ]:
# Focus on 0p7 threshold (used for training)
df07 = summary[summary["similarity_threshold"] == "0p7"].copy()

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Total samples per split
axes[0].bar(df07["split"], df07["n_total"] / 1e6,
            color=[SPLIT_COLORS.get(s, "grey") for s in df07["split"]])
axes[0].set_title("Total samples per split")
axes[0].set_ylabel("Millions")

# Active rate per split
axes[1].bar(df07["split"], df07["pct_active"],
            color=[SPLIT_COLORS.get(s, "grey") for s in df07["split"]])
axes[1].set_title("Active rate (%) per split")
axes[1].set_ylabel("%")
axes[1].axhline(5, ls="--", color="red", alpha=0.5, label="5% baseline")
axes[1].legend()

# Unique proteins per split
axes[2].bar(df07["split"], df07["n_proteins"],
            color=[SPLIT_COLORS.get(s, "grey") for s in df07["split"]])
axes[2].set_title("Unique proteins per split")
axes[2].set_ylabel("Count")

fig.suptitle("Dataset overview — 0p7 similarity threshold", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Active vs decoy stacked bar across ALL thresholds
fig, ax = plt.subplots(figsize=FIGSIZE)
pivot = summary.pivot_table(index=["similarity_threshold", "split"],
                             values=["n_active", "n_total"]).reset_index()
pivot["n_decoy"] = pivot["n_total"] - pivot["n_active"]
pivot["label"] = pivot["similarity_threshold"] + " / " + pivot["split"]

x = range(len(pivot))
ax.bar(x, pivot["n_decoy"] / 1e6, label="Decoy", color="#4C72B0", alpha=0.8)
ax.bar(x, pivot["n_active"] / 1e6, bottom=pivot["n_decoy"] / 1e6,
       label="Active", color="#DD8452", alpha=0.9)
ax.set_xticks(list(x))
ax.set_xticklabels(pivot["label"], rotation=45, ha="right", fontsize=9)
ax.set_ylabel("Millions")
ax.set_title("Active vs Decoy counts — all thresholds & splits")
ax.legend()
plt.tight_layout()
plt.show()

## 2  Ligand Chemistry

In [ ]:
# Sample for heavy computation — use full unique SMILES set if memory allows
SAMPLE_N = 50_000
smiles_sample = reg["smiles"].dropna().drop_duplicates().sample(SAMPLE_N, random_state=42)

from rdkit import Chem
from rdkit.Chem import Descriptors, AllChem

mols, mw_list, logp_list, hbd_list, hba_list, tpsa_list, rot_list, fp_density = [], [], [], [], [], [], [], []

for smi in smiles_sample:
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        continue
    mw_list.append(Descriptors.MolWt(mol))
    logp_list.append(Descriptors.MolLogP(mol))
    hbd_list.append(Descriptors.NumHDonors(mol))
    hba_list.append(Descriptors.NumHAcceptors(mol))
    tpsa_list.append(Descriptors.TPSA(mol))
    rot_list.append(Descriptors.NumRotatableBonds(mol))
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=2048)
    fp_density.append(fp.GetNumOnBits() / 2048)

chem_df = pd.DataFrame({
    "MolWt": mw_list, "LogP": logp_list, "HBD": hbd_list,
    "HBA": hba_list, "TPSA": tpsa_list, "RotBonds": rot_list,
    "FP_density": fp_density
})
print(f"Valid molecules: {len(chem_df):,} / {SAMPLE_N:,}")
chem_df.describe().round(2)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

props = [
    ("MolWt",      "Molecular Weight (Da)",  (0, 1000)),
    ("LogP",       "LogP",                   (-5, 10)),
    ("TPSA",       "TPSA (Å²)",              (0, 200)),
    ("HBD",        "H-Bond Donors",          (-0.5, 10)),
    ("RotBonds",   "Rotatable Bonds",        (-0.5, 20)),
    ("FP_density", "Morgan FP bit density",  (0, 0.3)),
]

for ax, (col, label, xlim) in zip(axes, props):
    data = chem_df[col].clip(*xlim)
    ax.hist(data, bins=50, color="#4C72B0", alpha=0.8, edgecolor="white", linewidth=0.3)
    ax.set_xlabel(label)
    ax.set_ylabel("Count")
    ax.set_xlim(xlim)
    med = data.median()
    ax.axvline(med, color="#DD8452", ls="--", lw=1.5, label=f"median={med:.1f}")
    ax.legend(fontsize=8)

fig.suptitle(f"Ligand physicochemical properties (n={len(chem_df):,} sampled)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## 3  Model Performance

In [ ]:
report = pd.read_csv(REPORT_CSV)
report

In [ ]:
# ROC-AUC across train / val / test — grouped bar chart
models = report["model"].tolist()
x = np.arange(len(models))
width = 0.25

fig, ax = plt.subplots(figsize=FIGSIZE)
bars_train = ax.bar(x - width, report["train_roc_auc"], width, label="Train",
                    color=SPLIT_COLORS["train"], alpha=0.85)
bars_val   = ax.bar(x,         report["val_roc_auc"],   width, label="Val",
                    color=SPLIT_COLORS["val"],   alpha=0.85)
bars_test  = ax.bar(x + width, report["test_roc_auc"],  width, label="Test",
                    color=SPLIT_COLORS["test"],  alpha=0.85)

# Reference line: random
ax.axhline(0.5, color="black", ls=":", lw=1.2, label="Random (0.5)")

# Annotate bars
for bars in [bars_train, bars_val, bars_test]:
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2, h + 0.005,
                f"{h:.3f}", ha="center", va="bottom", fontsize=7.5)

ax.set_xticks(x)
ax.set_xticklabels([m.replace("_", "\n") for m in models])
ax.set_ylabel("ROC-AUC")
ax.set_ylim(0, 1.05)
ax.set_title("ROC-AUC by model and split (0p7 similarity threshold)", fontweight="bold")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Average Precision (PR-AUC) — same layout
fig, ax = plt.subplots(figsize=FIGSIZE)
ax.bar(x - width, report["train_avg_precision"], width, label="Train",
       color=SPLIT_COLORS["train"], alpha=0.85)
ax.bar(x,         report["val_avg_precision"],   width, label="Val",
       color=SPLIT_COLORS["val"],   alpha=0.85)
ax.bar(x + width, report["test_avg_precision"],  width, label="Test",
       color=SPLIT_COLORS["test"],  alpha=0.85)

# Baseline: random classifier AP ≈ prevalence (~6%)
prevalence = 0.061
ax.axhline(prevalence, color="black", ls=":", lw=1.2, label=f"Random AP (≈{prevalence:.2f})")

ax.set_xticks(x)
ax.set_xticklabels([m.replace("_", "\n") for m in models])
ax.set_ylabel("Average Precision (PR-AUC)")
ax.set_ylim(0, 0.35)
ax.set_title("Average Precision by model and split", fontweight="bold")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Radar / spider chart — per-model multi-metric on TEST set
from matplotlib.patches import FancyArrowPatch

metrics = ["test_roc_auc", "test_avg_precision", "test_f1_score",
           "test_recall", "test_precision", "test_accuracy"]
labels  = ["ROC-AUC", "Avg Precision", "F1", "Recall", "Precision", "Accuracy"]

# Normalise: roc_auc & accuracy centred on 0.5, rest on 0
def normalise(col, val):
    if col in ("test_roc_auc", "test_accuracy"):
        return max(0, (val - 0.5) / 0.5)  # 0 = random, 1 = perfect
    return val

N = len(metrics)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(6, 6), subplot_kw=dict(polar=True))

for _, row in report.iterrows():
    vals = [normalise(m, row[m]) for m in metrics] + [normalise(metrics[0], row[metrics[0]])]
    color = COLORS.get(row["model"], "grey")
    ax.plot(angles, vals, color=color, lw=2, label=row["model"])
    ax.fill(angles, vals, color=color, alpha=0.12)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(labels, fontsize=10)
ax.set_title("Test-set metrics (normalised)\nROC-AUC & Accuracy: 0=random, 1=perfect",
             fontsize=10, pad=20)
ax.legend(loc="upper right", bbox_to_anchor=(1.35, 1.1))
plt.tight_layout()
plt.show()

## 4  Generalization Analysis

In [ ]:
# Melt to long form for easier plotting
long_rows = []
for _, row in report.iterrows():
    for split in ["train", "val", "test"]:
        long_rows.append({
            "model": row["model"],
            "split": split,
            "roc_auc":       row[f"{split}_roc_auc"],
            "avg_precision": row[f"{split}_avg_precision"],
            "f1":            row[f"{split}_f1_score"],
            "recall":        row[f"{split}_recall"],
        })
long_df = pd.DataFrame(long_rows)
long_df["split"] = pd.Categorical(long_df["split"], ["train", "val", "test"])

In [ ]:
# ROC-AUC learning curve (train→val→test) per model
fig, ax = plt.subplots(figsize=FIGSIZE)

for model, grp in long_df.groupby("model"):
    grp = grp.sort_values("split")
    color = COLORS.get(model, "grey")
    ax.plot(grp["split"], grp["roc_auc"], marker="o", lw=2.5, color=color, label=model)
    for _, r in grp.iterrows():
        ax.annotate(f"{r['roc_auc']:.3f}",
                    (r["split"], r["roc_auc"]),
                    textcoords="offset points", xytext=(0, 8),
                    ha="center", fontsize=8, color=color)

ax.axhline(0.5, color="black", ls=":", lw=1.2, label="Random")
ax.set_ylabel("ROC-AUC")
ax.set_ylim(0.2, 1.0)
ax.set_title("Generalization curve: Train → Val → Test", fontweight="bold")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Generalization gap: train_auc − test_auc
report["gen_gap"] = report["train_roc_auc"] - report["test_roc_auc"]

fig, ax = plt.subplots(figsize=(7, 4))
colors = [COLORS.get(m, "grey") for m in report["model"]]
bars = ax.barh(report["model"], report["gen_gap"], color=colors, alpha=0.85)
for bar, val in zip(bars, report["gen_gap"]):
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height() / 2,
            f"{val:.3f}", va="center", fontsize=10)
ax.set_xlabel("Generalization gap (Train ROC-AUC − Test ROC-AUC)")
ax.set_title("Overfitting severity (0p7 similarity split)", fontweight="bold")
ax.set_xlim(0, 0.7)
plt.tight_layout()
plt.show()

In [ ]:
# Heatmap: all metrics × all splits
metric_cols = [
    "train_roc_auc", "val_roc_auc", "test_roc_auc",
    "train_avg_precision", "val_avg_precision", "test_avg_precision",
    "train_f1_score", "val_f1_score", "test_f1_score",
    "train_recall", "val_recall", "test_recall",
]
hm_data = report.set_index("model")[metric_cols]

fig, ax = plt.subplots(figsize=(13, 3.5))
sns.heatmap(hm_data, annot=True, fmt=".3f", cmap="RdYlGn",
            vmin=0, vmax=1, linewidths=0.5, ax=ax,
            cbar_kws={"label": "metric value"})
ax.set_title("All metrics across train / val / test splits", fontweight="bold")
ax.set_xticklabels(ax.get_xticklabels(), rotation=35, ha="right", fontsize=9)
plt.tight_layout()
plt.show()

## 5  Virtual-Screening Metrics: Enrichment Factor & BEDROC

- **EF(χ%)**: ratio of actives found in top χ% vs random. EF=1 → random; EF=1/χ → perfect.
  Fractions evaluated: 0.1%, 0.2%, 0.5%, 1%, 2%, 5%, 10%, 15%, 20%
- **BEDROC(α)**: exponentially weights early-ranked actives (Truchon & Bayly 2007).
  α=20 ≈ top 8% weighted; α=80 ≈ top 2%; α=160 ≈ top 1%.

**Experiments**:
- *1D-VS*: 1D ligand similarity split with decoys → full VS evaluation (EF/BEDROC valid)
- *2D-Gen*: protein-cluster × ligand-similarity split → protein generalization test (actives only)

In [ ]:
# Load pre-computed VS metrics
vs_data = {}
for model_name in ["random_forest", "gradient_boosting", "svm"]:
    p = MODELS_DIR / f"{model_name}_vs_metrics.json"
    if p.exists():
        with open(p) as f:
            vs_data[model_name] = json.load(f)
        print(f"{model_name}: loaded")
    else:
        print(f"{model_name}: not found — run evaluate_vs_metrics.py first")

# Build a flat DataFrame
vs_rows = []
for model, d in vs_data.items():
    row = {"model": model}
    row.update(d["metrics"])
    row["prevalence"] = d["prevalence"]
    vs_rows.append(row)
vs_df = pd.DataFrame(vs_rows)
vs_df

In [ ]:
# Enrichment Factor at multiple levels
ef_cols = [c for c in vs_df.columns if c.startswith("ef_")]

def ef_col_to_label(col):
    s = col.replace("ef_", "").replace("pct", "")
    v = float(s.replace("p", ".")) if "p" in s else float(s)
    return f"{v}%"

ef_labels = [ef_col_to_label(c) for c in ef_cols]

fig, ax = plt.subplots(figsize=(13, 5))
x = np.arange(len(ef_cols))
width = 0.25
offsets = np.linspace(-width, width, len(vs_df))

for i, (_, row) in enumerate(vs_df.iterrows()):
    color = COLORS.get(row["model"], "grey")
    ax.bar(x + offsets[i], [row[c] for c in ef_cols],
           width * 0.9, label=row["model"], color=color, alpha=0.85)

ax.axhline(1.0, color="black", ls=":", lw=1.5, label="Random (EF=1)")
ax.set_xticks(x)
ax.set_xticklabels(ef_labels)
ax.set_ylabel("Enrichment Factor")
ax.set_xlabel("Top-χ fraction screened")
ax.set_title("Enrichment Factor at different screening fractions (test set)", fontweight="bold")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# EF enrichment curve — log x-axis shows early recognition clearly
fig, ax = plt.subplots(figsize=(10, 5))

def ef_col_to_frac_pct(col):
    s = col.replace("ef_", "").replace("pct", "")
    return float(s.replace("p", ".")) if "p" in s else float(s)

fracs_pct = [ef_col_to_frac_pct(c) for c in ef_cols]

for _, row in vs_df.iterrows():
    color = COLORS.get(row["model"], "grey")
    ef_vals = [row[c] for c in ef_cols]
    ax.plot(fracs_pct, ef_vals, marker="o", lw=2, color=color, label=row["model"])
    for frac, val in zip(fracs_pct, ef_vals):
        ax.annotate(f"{val:.2f}", (frac, val),
                    textcoords="offset points", xytext=(0, 6),
                    ha="center", fontsize=7, color=color)

ax.axhline(1.0, color="black", ls=":", lw=1.5, label="Random")
ax.set_xscale("log")
ax.set_xlabel("Top-χ% screened (log scale)")
ax.set_ylabel("Enrichment Factor")
ax.set_title("Enrichment curve — actives recovered vs random", fontweight="bold")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
bedroc_cols = [c for c in vs_df.columns if c.startswith("bedroc_")]
bedroc_labels = {
    "bedroc_a20":  "BEDROC a=20 (top ~8%)",
    "bedroc_a80":  "BEDROC a=80 (top ~2%)",
    "bedroc_a160": "BEDROC a=160 (top ~1%)",
}

fig, axes = plt.subplots(1, len(bedroc_cols), figsize=(5 * len(bedroc_cols), 4))
if len(bedroc_cols) == 1: axes = [axes]

for ax, col in zip(axes, bedroc_cols):
    bar_colors = [COLORS.get(m, "grey") for m in vs_df["model"]]
    bars = ax.bar(vs_df["model"], vs_df[col], color=bar_colors, alpha=0.85)
    for bar, val in zip(bars, vs_df[col]):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.001,
                f"{val:.4f}", ha="center", va="bottom", fontsize=9)
    prevalence = vs_df["prevalence"].mean()
    ax.axhline(prevalence, color="black", ls=":", lw=1.5,
               label=f"~Random ({prevalence:.3f})")
    ax.set_title(bedroc_labels.get(col, col), fontweight="bold")
    ax.set_ylabel("BEDROC")
    ax.set_ylim(0, max(float(vs_df[col].max()) * 1.3, 0.15))
    ax.set_xticklabels([m.replace("_", " ") for m in vs_df["model"]])
    ax.legend(fontsize=8)

fig.suptitle("BEDROC — early recognition of actives in ranked list", fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
display_cols = ["model"] + ef_cols + bedroc_cols
col_labels = ["Model"] + ef_labels + [bedroc_labels.get(c, c) for c in bedroc_cols]
summary_tbl = vs_df[display_cols].copy()
summary_tbl.columns = col_labels
summary_tbl = summary_tbl.set_index("Model")

fig, ax = plt.subplots(figsize=(16, 2.5))
sns.heatmap(summary_tbl.astype(float), annot=True, fmt=".3f",
            cmap="YlOrRd", linewidths=0.5, ax=ax,
            cbar_kws={"label": "metric value"})
ax.set_title("VS metrics heatmap — higher is better", fontweight="bold")
ax.set_xticklabels(ax.get_xticklabels(), rotation=35, ha="right")
plt.tight_layout()
plt.show()

print("\nRaw values:")
print(summary_tbl.round(4).to_string())

## 5  Training Efficiency

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Training time
axes[0].barh(report["model"], report["training_time_s"] / 60,
             color=[COLORS.get(m, "grey") for m in report["model"]], alpha=0.85)
axes[0].set_xlabel("Training time (minutes)")
axes[0].set_title("Training time")
for i, (_, row) in enumerate(report.iterrows()):
    axes[0].text(row["training_time_s"] / 60 + 0.3, i, f"{row['training_time_s']/60:.1f} min",
                 va="center", fontsize=9)

# ROC-AUC per minute of training
report["auc_per_min"] = report["test_roc_auc"] / (report["training_time_s"] / 60)
axes[1].barh(report["model"], report["auc_per_min"],
             color=[COLORS.get(m, "grey") for m in report["model"]], alpha=0.85)
axes[1].set_xlabel("Test ROC-AUC per minute of training")
axes[1].set_title("Efficiency (AUC / training-min)")

plt.suptitle("Training efficiency comparison", fontweight="bold")
plt.tight_layout()
plt.show()

## 6  PDBbind Comparison

In [ ]:
lit = pd.DataFrame([
    {"method": "RF + ECFP4 (lit.)",    "auc_lo": 0.72, "auc_hi": 0.78},
    {"method": "GBM + ECFP (lit.)",    "auc_lo": 0.74, "auc_hi": 0.80},
    {"method": "SVM + ECFP (lit.)",    "auc_lo": 0.70, "auc_hi": 0.76},
    {"method": "DeepDTA CNN (lit.)",    "auc_lo": 0.74, "auc_hi": 0.78},
])

fig, ax = plt.subplots(figsize=(10, 5))

# Literature ranges (error bars)
for i, row in lit.iterrows():
    mid = (row["auc_lo"] + row["auc_hi"]) / 2
    err = (row["auc_hi"] - row["auc_lo"]) / 2
    ax.errorbar(mid, row["method"], xerr=err, fmt="s",
                color="#7f7f7f", markersize=8, lw=2, capsize=5)

# Our results
model_labels = {"random_forest": "RF (ours, 0p7)",
                "gradient_boosting": "GBM (ours, 0p7)",
                "svm": "SVM (ours, 0p7)"}
for _, row in report.iterrows():
    color = COLORS.get(row["model"], "grey")
    label = model_labels.get(row["model"], row["model"])
    ax.scatter(row["test_roc_auc"], label, marker="D",
               color=color, s=80, zorder=5, label=label)
    ax.annotate(f"{row['test_roc_auc']:.3f}",
                (row["test_roc_auc"], label),
                xytext=(-8, -12), textcoords="offset points",
                fontsize=8, color=color)

ax.axvline(0.5, color="black", ls=":", lw=1, label="Random")
ax.set_xlabel("Test ROC-AUC")
ax.set_xlim(0.25, 0.95)
ax.set_title("Our results vs PDBbind literature baselines\n"
             "(gap expected: literature uses random/loose splits; we use 0p7 similarity split)",
             fontweight="bold")
ax.legend(loc="lower right", fontsize=8)
ax.grid(axis="x", alpha=0.4)
plt.tight_layout()
plt.show()

## 7  Feature Cache Stats

In [ ]:
import h5py

cache_path = ROOT / "training_data_full/feature_cache/morgan_r2_b2048.h5"
with h5py.File(cache_path, "r") as f:
    n_cached = len(f["key_hashes"])
    feat_shape = f["features"].shape
    config = json.loads(f.attrs["config"])
    packed = f.attrs["packed_bits"]

size_mb = cache_path.stat().st_size / 1e6
naive_mb = n_cached * 2048 * 4 / 1e6  # float32

print(f"Cached entries : {n_cached:,}")
print(f"Feature shape  : {feat_shape}  (packed bits)")
print(f"Config         : {config}")
print(f"Cache file     : {size_mb:.1f} MB")
print(f"Naive float32  : {naive_mb:.0f} MB")
print(f"Compression    : {naive_mb/size_mb:.0f}×")

## 9  Experiment Comparison

Unified view of all experiments with both ROC-AUC and VS-specific metrics.
New training runs will appear here automatically once their summaries are saved.

In [ ]:
# Load all training summaries from 1D and 2D experiments
exp_rows = []

def _load_dir(models_dir, split_label):
    for p in sorted(models_dir.glob("*_training_summary.json")):
        model_name = p.stem.replace("_training_summary", "")
        with open(p) as f:
            s = json.load(f)
        h = s["training_history"]
        train_m = h.get("train_metrics", {})
        val_m   = h.get("val_metrics", {})
        test_m  = h.get("test_metrics", {})
        train_auc = train_m.get("roc_auc", float("nan"))
        test_auc  = test_m.get("roc_auc", float("nan"))
        row = {
            "model":      model_name,
            "exp_label":  f"{model_name.replace('_',' ')} ({split_label})",
            "split":      split_label,
            "train_auc":  train_auc,
            "val_auc":    val_m.get("roc_auc", float("nan")),
            "test_auc":   test_auc,
            "gen_gap":    train_auc - test_auc,
            "train_time": h.get("training_time", float("nan")),
        }
        # VS metrics only for 1D (2D has no decoys)
        vs_p = models_dir / f"{model_name}_vs_metrics.json"
        if vs_p.exists():
            with open(vs_p) as f:
                vs = json.load(f)
            for k in ["ef_0.1pct", "ef_1pct", "ef_5pct",
                      "bedroc_a20", "bedroc_a80", "bedroc_a160"]:
                row[k] = vs["metrics"].get(k, float("nan"))
        exp_rows.append(row)

_load_dir(MODELS_DIR,    "1d")
_load_dir(MODELS_DIR_2D, "2d")

exp_df = pd.DataFrame(exp_rows)
display_cols = [c for c in ["exp_label", "split", "train_auc", "val_auc", "test_auc",
                             "gen_gap", "ef_1pct", "ef_5pct",
                             "bedroc_a20", "bedroc_a80", "bedroc_a160"]
                if c in exp_df.columns]
exp_df[display_cols].round(4)


In [ ]:
# ROC-AUC + EF@1% + BEDROC — bars grouped by experiment, hatched for 2D
metrics_to_plot = [
    ("test_auc",    "Test ROC-AUC"),
    ("gen_gap",     "Gen. Gap (train-test AUC)"),
    ("ef_1pct",     "EF @ 1%"),
    ("bedroc_a80",  "BEDROC a=80"),
]
valid = [(col, lbl) for col, lbl in metrics_to_plot if col in exp_df.columns]
fig, axes = plt.subplots(1, len(valid), figsize=(4.5 * len(valid), 5))
if len(valid) == 1:
    axes = [axes]

for ax, (col, label) in zip(axes, valid):
    for i, (_, row) in enumerate(exp_df.iterrows()):
        color   = COLORS.get(row["model"], "grey")
        hatch   = SPLIT_HATCHES.get(row["split"], "")
        val     = row[col] if not pd.isna(row.get(col, float("nan"))) else 0
        bar = ax.bar(i, val, color=color, hatch=hatch, edgecolor="white",
                     alpha=0.85, linewidth=0.5)
    ax.set_xticks(range(len(exp_df)))
    ax.set_xticklabels(
        [r["exp_label"] for _, r in exp_df.iterrows()],
        fontsize=7, rotation=40, ha="right")
    ax.set_title(label, fontweight="bold")
    if col in ("test_auc", "val_auc"):
        ax.axhline(0.5, color="black", ls=":", lw=1, label="Random")
        ax.legend(fontsize=7)
    elif col.startswith("ef_"):
        ax.axhline(1.0, color="black", ls=":", lw=1, label="Random")
        ax.legend(fontsize=7)

# Legend: model colors + split hatches
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor=COLORS["random_forest"],    label="Random Forest"),
    Patch(facecolor=COLORS["gradient_boosting"], label="GBM"),
    Patch(facecolor=COLORS["svm"],              label="SVM"),
    Patch(facecolor="grey", hatch="",  edgecolor="black", label="1D split"),
    Patch(facecolor="grey", hatch="//", edgecolor="black", label="2D split"),
]
fig.legend(handles=legend_elements, loc="upper right", fontsize=8,
           bbox_to_anchor=(1.0, 1.0), framealpha=0.9)
fig.suptitle("All experiments — 1D (solid) vs 2D (hatched) split", fontweight="bold")
plt.tight_layout()
plt.show()


In [ ]:
# Scatter: test ROC-AUC vs EF@1%, and gen-gap vs BEDROC a=80
# Both 1D and 2D experiments now have EF/BEDROC (decoys included for all test partitions)
plot_df = exp_df.dropna(subset=["ef_1pct", "bedroc_a80"])

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: test AUC vs EF@1%
ax = axes[0]
for _, row in plot_df.iterrows():
    color  = COLORS.get(row["model"], "grey")
    marker = SPLIT_MARKERS.get(row["split"], "o")
    ax.scatter(row["test_auc"], row["ef_1pct"], s=160, color=color,
               marker=marker, zorder=5, edgecolors="black", linewidths=0.6)
    ax.annotate(row["model"].replace("_", " ") + "\n" + row["split"],
                (row["test_auc"], row["ef_1pct"]),
                textcoords="offset points", xytext=(5, 4), fontsize=7, color=color)
ax.axhline(1.0, color="black", ls=":", lw=1, label="EF random")
ax.axvline(0.5, color="grey",  ls=":", lw=1, label="AUC random")
ax.set_xlabel("Test ROC-AUC")
ax.set_ylabel("EF @ 1%")
ax.set_title("Discrimination vs early enrichment", fontweight="bold")
ax.legend(fontsize=8)

# Right: gen-gap vs BEDROC a=80
ax = axes[1]
for _, row in plot_df.iterrows():
    color  = COLORS.get(row["model"], "grey")
    marker = SPLIT_MARKERS.get(row["split"], "o")
    ax.scatter(row["gen_gap"], row["bedroc_a80"], s=160, color=color,
               marker=marker, zorder=5, edgecolors="black", linewidths=0.6)
    ax.annotate(row["model"].replace("_", " ") + "\n" + row["split"],
                (row["gen_gap"], row["bedroc_a80"]),
                textcoords="offset points", xytext=(5, 4), fontsize=7, color=color)
ax.axvline(0, color="black", ls=":", lw=0.8)
ax.set_xlabel("Generalization gap (train - test AUC)")
ax.set_ylabel("BEDROC a=80")
ax.set_title("Overfitting vs early-recognition quality", fontweight="bold")

from matplotlib.lines import Line2D
leg = [Line2D([0],[0], marker='o', color='grey', ms=8, label='1D split', ls='none'),
       Line2D([0],[0], marker='s', color='grey', ms=8, label='2D split', ls='none')]
ax.legend(handles=leg, fontsize=8)

fig.suptitle("Key trade-offs: 1D (circle) vs 2D (square)", fontweight="bold")
plt.tight_layout()
plt.show()


In [ ]:
# Full metric heatmap across all experiments
hm_cols = [c for c in ["train_auc", "val_auc", "test_auc", "gen_gap",
                        "ef_1pct", "ef_5pct", "bedroc_a20", "bedroc_a80", "bedroc_a160"]
           if c in exp_df.columns]
hm = exp_df.set_index("exp_label")[hm_cols].fillna(float("nan"))

n_rows = len(hm)
fig, ax = plt.subplots(figsize=(14, max(3, n_rows * 0.9)))
sns.heatmap(hm.astype(float), annot=True, fmt=".3f", cmap="RdYlGn",
            vmin=0, vmax=1, linewidths=0.5, ax=ax,
            mask=hm.isna())  # grey-out NaN cells (2D experiments have no EF/BEDROC)
ax.set_title("All experiments — full metric heatmap (grey = N/A for 2D split)",
             fontweight="bold")
ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha="right")
ax.set_yticklabels(ax.get_yticklabels(), rotation=0)
plt.tight_layout()
plt.show()


## 10  1D vs 2D Generalization Comparison

Side-by-side comparison of how each model degrades when moving from a 1D split
(ligand similarity only) to a 2D split (novel protein family **and** novel ligand).

- **1D test**: ligands with <0.7 Tanimoto to training ligands (ligand novelty only); large decoy set (~94% decoys)
- **2D test**: held-out protein cluster x ligand split=test; protein-matched decoys included (~6.8% active rate)
- EF and BEDROC are computed for **both** splits — decoys are always included via protein matching


In [ ]:
# Compare 1D vs 2D test ROC-AUC per model
auc_1d = exp_df[exp_df["split"] == "1d"].set_index("model")["test_auc"]
auc_2d = exp_df[exp_df["split"] == "2d"].set_index("model")["test_auc"]
common_models = sorted(set(auc_1d.index) & set(auc_2d.index))

x = np.arange(len(common_models))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 5))
bars1 = ax.bar(x - width/2, [auc_1d[m] for m in common_models],
               width, label="1D split (ligand novelty)",
               color=[COLORS[m] for m in common_models], alpha=0.85)
bars2 = ax.bar(x + width/2, [auc_2d[m] for m in common_models],
               width, label="2D split (protein + ligand novelty)",
               color=[COLORS[m] for m in common_models], alpha=0.85, hatch="//",
               edgecolor="white")
ax.axhline(0.5, color="black", ls=":", lw=1.2, label="Random")
ax.set_xticks(x)
ax.set_xticklabels([m.replace("_", " ") for m in common_models], fontsize=11)
ax.set_ylabel("Test ROC-AUC")
ax.set_title("Generalization collapse: 1D (solid) vs 2D (hatched) test AUC",
             fontweight="bold")
ax.set_ylim(0, 1)
ax.legend(fontsize=9)
for bar in list(bars1) + list(bars2):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.012,
            f"{bar.get_height():.3f}", ha="center", va="bottom", fontsize=8)
plt.tight_layout()
plt.show()


In [ ]:
# Generalization drop: 1D AUC - 2D AUC per model
drop = pd.DataFrame({
    "model":    common_models,
    "auc_1d":   [auc_1d[m] for m in common_models],
    "auc_2d":   [auc_2d[m] for m in common_models],
    "drop":     [auc_1d[m] - auc_2d[m] for m in common_models],
    "gen_1d":   [exp_df[(exp_df.model==m)&(exp_df.split=="1d")]["gen_gap"].values[0] for m in common_models],
    "gen_2d":   [exp_df[(exp_df.model==m)&(exp_df.split=="2d")]["gen_gap"].values[0] for m in common_models],
})

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left: AUC drop bar
colors = [COLORS[m] for m in drop["model"]]
axes[0].bar(drop["model"].str.replace("_", " "), drop["drop"], color=colors, alpha=0.85)
axes[0].axhline(0, color="black", lw=0.8)
axes[0].set_ylabel("AUC drop (1D test - 2D test)")
axes[0].set_title("AUC drop from 1D to 2D split", fontweight="bold")
axes[0].set_xticklabels(drop["model"].str.replace("_", " "), rotation=15, ha="right")
for i, v in enumerate(drop["drop"]):
    axes[0].text(i, v + 0.003, f"{v:.3f}", ha="center", fontsize=9)

# Right: gen gap comparison
x2 = np.arange(len(common_models))
w2 = 0.35
axes[1].bar(x2 - w2/2, drop["gen_1d"], w2, color=colors, alpha=0.85, label="1D gen gap")
axes[1].bar(x2 + w2/2, drop["gen_2d"], w2, color=colors, alpha=0.85, hatch="//",
            edgecolor="white", label="2D gen gap")
axes[1].set_xticks(x2)
axes[1].set_xticklabels([m.replace("_", " ") for m in common_models], rotation=15, ha="right")
axes[1].set_ylabel("Generalization gap (train - test AUC)")
axes[1].set_title("Generalization gap: 1D vs 2D", fontweight="bold")
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()
print(drop[["model","auc_1d","auc_2d","drop","gen_1d","gen_2d"]].round(4).to_string(index=False))


## 11  GNINA Docking vs ML Models — Per-Target Comparison

GNINA was run on 15 targets; 10 have both actives and decoys (EF/BEDROC computable).
Methods compared:
- **GNINA CNN_VS** — CNN virtual-screening score (higher = more active)
- **GNINA -Affinity** — negated minimizedAffinity (more negative kcal/mol = better binder)
- **RF / GBM / SVM** — 1D split models, evaluated on each target's test-split ligands

Note: ML models use random protein embeddings so per-target performance reflects
ligand-based enrichment only (same protein embedding for all ligands of a target).


In [ ]:
# Load GNINA docking metrics
DOCKING_METRICS_DIR = ROOT / 'benchmarks/04_docking/results/docking_metrics'
ML_PT_DIR           = ROOT / 'benchmarks/04_docking/results/ml_per_target_metrics'

with open(DOCKING_METRICS_DIR / 'all_targets_docking_metrics.json') as f:
    gnina_all = json.load(f)

with open(ML_PT_DIR / 'ml_per_target_metrics.json') as f:
    ml_all = json.load(f)

GNINA_METHODS = {
    'GNINA CNN_VS':   ('CNN_VS',            'ef_1pct', 'bedroc_a80', 'roc_auc'),
    'GNINA -Affinity':('minimizedAffinity',  'ef_1pct', 'bedroc_a80', 'roc_auc'),
}
ML_METHODS = ['random_forest', 'gradient_boosting', 'svm']

# Valid targets = those with GNINA CNN_VS results (has n_active key)
valid_targets = sorted([
    u for u, d in gnina_all.items()
    if d['status'] == 'ok' and 'n_active' in d.get('CNN_VS', {})
])
print(f'Valid targets (actives + decoys): {valid_targets}')
print(f'n = {len(valid_targets)}')


In [ ]:
# Build EF@1% matrix: rows=targets, cols=methods
all_methods = list(GNINA_METHODS.keys()) + ML_METHODS
method_labels = {
    'GNINA CNN_VS':      'GNINA\nCNN_VS',
    'GNINA -Affinity':   'GNINA\n-Affinity',
    'random_forest':     'RF',
    'gradient_boosting': 'GBM',
    'svm':               'SVM',
}

def get_gnina_metric(uniprot, score_key, metric_key):
    d = gnina_all.get(uniprot, {})
    if d.get('status') != 'ok': return float('nan')
    return d.get(score_key, {}).get(metric_key, float('nan'))

def get_ml_metric(model_name, uniprot, metric_key):
    d = ml_all.get(model_name, {}).get(uniprot, {})
    if d.get('status') != 'ok': return float('nan')
    return d.get('metrics', {}).get(metric_key, float('nan'))

for metric_key, metric_label in [('ef_1pct', 'EF @ 1%'), ('bedroc_a80', 'BEDROC a=80'), ('roc_auc', 'ROC-AUC')]:
    rows = {}
    for mname, (score_key, ef_k, b_k, auc_k) in GNINA_METHODS.items():
        rows[method_labels[mname]] = [get_gnina_metric(u, score_key, metric_key) for u in valid_targets]
    for mname in ML_METHODS:
        rows[method_labels[mname]] = [get_ml_metric(mname, u, metric_key) for u in valid_targets]

    hm = pd.DataFrame(rows, index=valid_targets).T

    fig, ax = plt.subplots(figsize=(max(10, len(valid_targets)*1.1), 4))
    if metric_key == 'ef_1pct':
        vmin, vmax, cmap = 0, 6, 'RdYlGn'
    elif metric_key == 'roc_auc':
        vmin, vmax, cmap = 0, 1, 'RdYlGn'
    else:
        vmin, vmax, cmap = 0, 1, 'RdYlGn'

    sns.heatmap(hm.astype(float), annot=True, fmt='.3f', cmap=cmap,
                vmin=vmin, vmax=vmax, linewidths=0.5, ax=ax,
                mask=hm.isna())
    ax.set_title(f'Per-target {metric_label} — GNINA vs ML models', fontweight='bold')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right', fontsize=9)
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=9)
    plt.tight_layout()
    plt.show()


In [ ]:
# Aggregated comparison: mean EF@1% and BEDROC across valid targets
summary_rows = []
for mname, (score_key, *_) in GNINA_METHODS.items():
    ef_vals = [get_gnina_metric(u, score_key, 'ef_1pct') for u in valid_targets]
    bd_vals = [get_gnina_metric(u, score_key, 'bedroc_a80') for u in valid_targets]
    au_vals = [get_gnina_metric(u, score_key, 'roc_auc') for u in valid_targets]
    summary_rows.append({
        'method': method_labels[mname],
        'mean_ef1': np.nanmean(ef_vals),
        'mean_bedroc80': np.nanmean(bd_vals),
        'mean_auc': np.nanmean(au_vals),
        'type': 'docking',
    })
for mname in ML_METHODS:
    ef_vals = [get_ml_metric(mname, u, 'ef_1pct') for u in valid_targets]
    bd_vals = [get_ml_metric(mname, u, 'bedroc_a80') for u in valid_targets]
    au_vals = [get_ml_metric(mname, u, 'roc_auc') for u in valid_targets]
    summary_rows.append({
        'method': method_labels[mname],
        'mean_ef1': np.nanmean(ef_vals),
        'mean_bedroc80': np.nanmean(bd_vals),
        'mean_auc': np.nanmean(au_vals),
        'type': 'ml',
    })

sum_df = pd.DataFrame(summary_rows)
print('Mean metrics across', len(valid_targets), 'targets:')
print(sum_df[['method','mean_auc','mean_ef1','mean_bedroc80']].round(3).to_string(index=False))

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
METHOD_COLORS = {
    'GNINA\nCNN_VS':   '#2ca02c',
    'GNINA\n-Affinity':'#98df8a',
    'RF':  COLORS['random_forest'],
    'GBM': COLORS['gradient_boosting'],
    'SVM': COLORS['svm'],
}
for ax, (col, label, ref) in zip(axes, [
    ('mean_auc',       'Mean ROC-AUC',   0.5),
    ('mean_ef1',       'Mean EF @ 1%',   1.0),
    ('mean_bedroc80',  'Mean BEDROC a=80', None),
]):
    colors = [METHOD_COLORS.get(m, 'grey') for m in sum_df['method']]
    ax.bar(sum_df['method'], sum_df[col], color=colors, alpha=0.85)
    if ref is not None:
        ax.axhline(ref, color='black', ls=':', lw=1.2)
    ax.set_title(label, fontweight='bold')
    ax.set_xticklabels(sum_df['method'], rotation=20, ha='right', fontsize=9)
    for i, v in enumerate(sum_df[col]):
        ax.text(i, v + 0.01, f'{v:.2f}', ha='center', fontsize=8)

fig.suptitle(f'GNINA vs ML — mean VS metrics over {len(valid_targets)} targets', fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# Scatter: GNINA CNN_VS EF@1% vs best-ML EF@1% per target
# Helps identify targets where docking wins vs ML wins
gnina_ef = {u: get_gnina_metric(u, 'CNN_VS', 'ef_1pct') for u in valid_targets}
best_ml_ef = {
    u: max(get_ml_metric(m, u, 'ef_1pct') for m in ML_METHODS)
    for u in valid_targets
}

fig, ax = plt.subplots(figsize=(7, 6))
xs = [gnina_ef[u] for u in valid_targets]
ys = [best_ml_ef[u] for u in valid_targets]
ax.scatter(xs, ys, s=120, color='steelblue', zorder=5)
for u, x, y in zip(valid_targets, xs, ys):
    ax.annotate(u, (x, y), textcoords='offset points', xytext=(5,4), fontsize=8)

# Diagonal: GNINA = best ML
lim = max(max(xs), max(ys)) * 1.1
ax.plot([0, lim], [0, lim], 'k--', lw=1, label='GNINA = best ML')
ax.axhline(1.0, color='grey', ls=':', lw=0.8)
ax.axvline(1.0, color='grey', ls=':', lw=0.8)
ax.set_xlabel('GNINA CNN_VS — EF @ 1%', fontsize=11)
ax.set_ylabel('Best ML (RF/GBM/SVM) — EF @ 1%', fontsize=11)
ax.set_title('Per-target: GNINA vs best ML early enrichment\n'
             'above diagonal = ML wins; below = GNINA wins', fontweight='bold')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()


## 12  Boltzina (Boltz-2 Affinity) vs GNINA vs ML — Per-Target Comparison

Boltzina docks ligands with AutoDock Vina, then rescores each protein-ligand complex
through the full Boltz-2 neural network (Pairformer trunk + affinity head).

**Scores compared:**
- `affinity_prob_binary` — Boltz-2 P(binder) probability (best Boltz-2 metric)
- `affinity_pred_value` — Boltz-2 normalized affinity
- `docking_score` — Vina energy (negated; shared between Boltzina and GNINA)
- GNINA `cnn_score` / `cnn_affinity` — 3D CNN rescoring
- ML models (RF, GBM, SVM) — fingerprint-based

In [ ]:
# ── Load Boltzina results ─────────────────────────────────────────────
BOLTZINA_DIR = ROOT / 'benchmarks/05_boltzina/results/raw_results'
GNINA_RESULTS_DIR = Path('/tmp/gnina_results')

# Targets with both boltzina AND gnina results
BOLTZ_TARGETS = ['P09211', 'Q13490']

from sklearn.metrics import roc_auc_score

def compute_metrics(scores, labels):
    """ROC-AUC, EF@1%, EF@5% from scores + binary labels."""
    n = len(scores); n_act = labels.sum()
    if n_act == 0 or n_act == n:
        return {'roc_auc': float('nan'), 'ef_1pct': float('nan'), 'ef_5pct': float('nan')}
    auc = roc_auc_score(labels, scores)
    n1 = max(1, int(np.ceil(n * 0.01)))
    top1 = np.argsort(scores)[::-1][:n1]
    ef1 = (labels[top1].sum() / n1) / (n_act / n)
    n5 = max(1, int(np.ceil(n * 0.05)))
    top5 = np.argsort(scores)[::-1][:n5]
    ef5 = (labels[top5].sum() / n5) / (n_act / n)
    return {'roc_auc': auc, 'ef_1pct': ef1, 'ef_5pct': ef5}

# Collect all metrics per target
all_results = {}
for uid in BOLTZ_TARGETS:
    all_results[uid] = {}

    # Boltzina (Vina docking + Boltz-2 scoring, predicted structure)
    boltz_csv = BOLTZINA_DIR / uid / 'boltzina_results.csv'
    if not boltz_csv.exists():
        # Fall back to comparison test dirs
        boltz_csv = Path(f'/tmp/boltz_cmpA/results/raw_results/{uid}/boltzina_results.csv')
    if boltz_csv.exists():
        df = pd.read_csv(boltz_csv)
        df['is_active'] = df['ligand_name'].str.contains('/actives/').astype(int)
        labels = df['is_active'].values
        all_results[uid]['Boltzina\nprob_binary'] = compute_metrics(
            df['affinity_probability_binary'].values, labels)
        all_results[uid]['Boltzina\npred_value'] = compute_metrics(
            df['affinity_pred_value'].values, labels)
        all_results[uid]['Boltzina\nVina score'] = compute_metrics(
            -df['docking_score'].values, labels)
        print(f'{uid} boltzina: {len(df)} ligands ({labels.sum()} actives)')

    # GNINA
    gnina_csv = GNINA_RESULTS_DIR / uid / 'gnina_results.csv'
    if gnina_csv.exists():
        df = pd.read_csv(gnina_csv)
        df['is_active'] = df['ligand_name'].str.contains('/actives/').astype(int)
        labels = df['is_active'].values
        all_results[uid]['GNINA\ncnn_score'] = compute_metrics(
            df['cnn_score'].values, labels)
        all_results[uid]['GNINA\ncnn_affinity'] = compute_metrics(
            df['cnn_affinity'].values, labels)
        all_results[uid]['GNINA\nVina score'] = compute_metrics(
            -df['vina_score'].values, labels)
        print(f'{uid} gnina: {len(df)} ligands ({labels.sum()} actives)')

    # ML per-target (if available)
    for mname, mlabel in [('random_forest', 'RF'), ('gradient_boosting', 'GBM'), ('svm', 'SVM')]:
        try:
            ml_met = ml_all.get(mname, {}).get(uid, {})
            if ml_met.get('status') == 'ok':
                m = ml_met['metrics']
                all_results[uid][mlabel] = {
                    'roc_auc': m.get('roc_auc', float('nan')),
                    'ef_1pct': m.get('ef_1pct', float('nan')),
                    'ef_5pct': m.get('ef_5pct', float('nan')),
                }
        except Exception:
            pass

print(f'\nTargets loaded: {list(all_results.keys())}')

In [ ]:
# ── Per-target heatmaps: ROC-AUC and EF@1% ───────────────────────────
METHOD_ORDER = [
    'Boltzina\nprob_binary', 'Boltzina\npred_value', 'Boltzina\nVina score',
    'GNINA\ncnn_score', 'GNINA\ncnn_affinity', 'GNINA\nVina score',
    'RF', 'GBM', 'SVM',
]
METHOD_COLORS_MAP = {
    'Boltzina\nprob_binary': '#9467bd',
    'Boltzina\npred_value':  '#c5b0d5',
    'Boltzina\nVina score':  '#d6bce4',
    'GNINA\ncnn_score':      '#2ca02c',
    'GNINA\ncnn_affinity':   '#98df8a',
    'GNINA\nVina score':     '#c7e9c0',
    'RF': COLORS['random_forest'],
    'GBM': COLORS['gradient_boosting'],
    'SVM': COLORS['svm'],
}

for metric_key, metric_label, vmin, vmax in [
    ('roc_auc', 'ROC-AUC', 0, 1),
    ('ef_1pct', 'EF @ 1%', 0, 8),
    ('ef_5pct', 'EF @ 5%', 0, 8),
]:
    rows = {}
    for m in METHOD_ORDER:
        vals = []
        for uid in BOLTZ_TARGETS:
            v = all_results.get(uid, {}).get(m, {}).get(metric_key, float('nan'))
            vals.append(v)
        rows[m] = vals
    hm = pd.DataFrame(rows, index=BOLTZ_TARGETS).T

    fig, ax = plt.subplots(figsize=(5, 5))
    sns.heatmap(hm.astype(float), annot=True, fmt='.3f', cmap='RdYlGn',
                vmin=vmin, vmax=vmax, linewidths=0.5, ax=ax,
                mask=hm.isna())
    ax.set_title(f'Per-target {metric_label} — Boltzina vs GNINA vs ML', fontweight='bold')
    ax.set_xticklabels(BOLTZ_TARGETS, rotation=0, fontsize=10)
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=9)
    plt.tight_layout()
    plt.show()

In [ ]:
# ── Aggregated bar chart: mean metrics across targets ─────────────────
summary_rows = []
for m in METHOD_ORDER:
    aucs = [all_results[u].get(m, {}).get('roc_auc', float('nan')) for u in BOLTZ_TARGETS]
    ef1s = [all_results[u].get(m, {}).get('ef_1pct', float('nan')) for u in BOLTZ_TARGETS]
    ef5s = [all_results[u].get(m, {}).get('ef_5pct', float('nan')) for u in BOLTZ_TARGETS]
    summary_rows.append({
        'method': m,
        'mean_auc': np.nanmean(aucs),
        'mean_ef1': np.nanmean(ef1s),
        'mean_ef5': np.nanmean(ef5s),
    })
sum_df = pd.DataFrame(summary_rows)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, (col, label, ref) in zip(axes, [
    ('mean_auc', f'Mean ROC-AUC (n={len(BOLTZ_TARGETS)} targets)', 0.5),
    ('mean_ef1', f'Mean EF @ 1%', 1.0),
    ('mean_ef5', f'Mean EF @ 5%', 1.0),
]):
    colors = [METHOD_COLORS_MAP.get(m, 'grey') for m in sum_df['method']]
    bars = ax.bar(range(len(sum_df)), sum_df[col], color=colors, alpha=0.85)
    if ref is not None:
        ax.axhline(ref, color='black', ls=':', lw=1.2, label='random' if ref == 0.5 else 'no enrichment')
    ax.set_title(label, fontweight='bold')
    ax.set_xticks(range(len(sum_df)))
    ax.set_xticklabels(sum_df['method'], rotation=35, ha='right', fontsize=8)
    for i, v in enumerate(sum_df[col]):
        if not np.isnan(v):
            ax.text(i, v + 0.02, f'{v:.2f}', ha='center', fontsize=7)
    ax.legend(fontsize=8)

fig.suptitle('Boltzina vs GNINA vs ML — Aggregated VS Metrics', fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Print summary table
print(sum_df[['method', 'mean_auc', 'mean_ef1', 'mean_ef5']].round(3).to_string(index=False))

In [ ]:
# ── Predicted vs Experimental structure comparison (Boltzina only) ────
BOLTZ_CMP_A = Path('/tmp/boltz_cmpA/results/raw_results')  # predicted structure
BOLTZ_CMP_B = Path('/tmp/boltz_cmpB/results/raw_results')  # experimental structure

struct_rows = []
for uid in BOLTZ_TARGETS:
    for cond_dir, cond_label in [(BOLTZ_CMP_A, 'Predicted'), (BOLTZ_CMP_B, 'Experimental')]:
        csv = cond_dir / uid / 'boltzina_results.csv'
        if not csv.exists():
            continue
        df = pd.read_csv(csv)
        df['is_active'] = df['ligand_name'].str.contains('/actives/').astype(int)
        labels = df['is_active'].values
        for col, direction, score_label in [
            ('affinity_probability_binary', 'higher', 'prob_binary'),
            ('affinity_pred_value', 'higher', 'pred_value'),
            ('docking_score', 'lower', 'docking_score'),
        ]:
            scores = -df[col].values if direction == 'lower' else df[col].values
            m = compute_metrics(scores, labels)
            struct_rows.append({
                'target': uid, 'structure': cond_label, 'score': score_label,
                'roc_auc': m['roc_auc'], 'ef_1pct': m['ef_1pct'],
            })

struct_df = pd.DataFrame(struct_rows)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, metric, label in zip(axes, ['roc_auc', 'ef_1pct'], ['ROC-AUC', 'EF @ 1%']):
    pivot = struct_df.pivot_table(index=['target', 'score'], columns='structure', values=metric)
    pivot = pivot.reindex(columns=['Predicted', 'Experimental'])
    pivot.plot(kind='bar', ax=ax, color=['#9467bd', '#ff7f0e'], alpha=0.85)
    ax.set_title(f'Boltz-2 {label}: Predicted vs Experimental Structure', fontweight='bold')
    ax.set_xlabel('')
    ax.set_xticklabels([f'{t}\n{s}' for t, s in pivot.index], rotation=30, ha='right', fontsize=8)
    if metric == 'roc_auc':
        ax.axhline(0.5, color='black', ls=':', lw=1)
    elif metric == 'ef_1pct':
        ax.axhline(1.0, color='black', ls=':', lw=1)
    ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# ── Score distribution: actives vs decoys per method ──────────────────
fig, axes = plt.subplots(2, 3, figsize=(16, 9))

for row, uid in enumerate(BOLTZ_TARGETS):
    # Boltzina
    boltz_csv = BOLTZINA_DIR / uid / 'boltzina_results.csv'
    if not boltz_csv.exists():
        boltz_csv = Path(f'/tmp/boltz_cmpA/results/raw_results/{uid}/boltzina_results.csv')
    bdf = pd.read_csv(boltz_csv)
    bdf['is_active'] = bdf['ligand_name'].str.contains('/actives/')

    # GNINA
    gdf = pd.read_csv(GNINA_RESULTS_DIR / uid / 'gnina_results.csv')
    gdf['is_active'] = gdf['ligand_name'].str.contains('/actives/')

    for col_idx, (df, col, label, color) in enumerate([
        (bdf, 'affinity_probability_binary', f'{uid}\nBoltz-2 prob_binary', '#9467bd'),
        (gdf, 'cnn_score', f'{uid}\nGNINA cnn_score', '#2ca02c'),
        (bdf, 'docking_score', f'{uid}\nVina docking_score', '#4C72B0'),
    ]):
        ax = axes[row, col_idx]
        vals_act = df.loc[df['is_active'], col].dropna()
        vals_dec = df.loc[~df['is_active'], col].dropna()
        ax.hist(vals_dec, bins=30, alpha=0.6, label=f'Decoys (n={len(vals_dec)})', color='grey', density=True)
        ax.hist(vals_act, bins=30, alpha=0.7, label=f'Actives (n={len(vals_act)})', color=color, density=True)
        ax.set_title(label, fontweight='bold', fontsize=10)
        ax.legend(fontsize=8)
        ax.set_ylabel('Density' if col_idx == 0 else '')

fig.suptitle('Score Distributions: Actives vs Decoys', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()